In [ ]:
# Install the PyPharao code and py3Dmol
!pip install rdkit
!pip install git+https://github.com/silicos-it/PyPharao.git
!pip install mols2grid py3Dmol

In [ ]:
# Import all required modules
from tqdm import tqdm_notebook as tqdm

from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem.Draw import IPythonConsole

from IPython.display import display
from IPython.display import SVG

import py3Dmol
from ipywidgets import widgets

import mols2grid
from pypharao import *

from pathlib import Path
import requests

In [ ]:
# Build the pharmacophore query from paracetamol
query_mol = Chem.AddHs(Chem.MolFromSmiles("CC(=O)Nc1ccc(O)cc1"))
AllChem.EmbedMolecule(query_mol)
AllChem.UFFOptimizeMolecule(query_mol)
query = query_pharmacophore_from_molecule(query_mol, name="paracetamol")
print(f"\nQuery {query.get_name()!r} ({len(query)} features):")
for p in query:
    print(f"  {p.type.value:<10} center={p.center}")

In [ ]:
view = py3Dmol.view(
    data=Chem.MolToMolBlock(query_mol),  # Convert the RDKit molecule for py3Dmol
    style={"stick": {}, "sphere": {"scale": 0.3}}
)
view.zoomTo()

In [ ]:
# Add excluded volumes around the query
n_excl = add_excluded_volume(
    query_mol,
    query,
    shell_inner=1.5,     # start 1.5 Å outside the vdW surface
    shell_outer=3.5,     # end   2.5 Å outside the vdW surface
    spacing=3.0,         # grid step (Å)
    feature_clearance=4.5,  # drop grid points within 4.5 Å of an existing feature
    # max_excl=0 by default (no cap); pass max_excl=512 etc. to limit count
)
print(f"\nAdded {n_excl} EXCL spheres around paracetamol "
      f"({len(query)} features total).")

# Persist the query (with its EXCL envelope) for inspection in a viewer.
OUT_PDB = Path("./paracetamol_with_excl.pdb")
query.write_pdb(OUT_PDB)
print(f"Wrote {OUT_PDB.name}")

In [ ]:
view = py3Dmol.view()
with open(Path("./paracetamol_with_excl.pdb"), 'r') as f: pdb_data = f.read()
view.addModel(pdb_data, 'pdb')
view.addModel(Chem.MolToMolBlock(query_mol), 'sdf')

view.setStyle({'model': 0}, {'line': {}, 'sphere': {'radius': 0.05}})
view.setStyle({'model': 1}, {'stick': {}, 'sphere': {'radius': 0.2}})

view.show()

In [ ]:
# Download a 10,000 compound dataset
MAX_COMPOUNDS = 2000   # None if all

# Define the URL of the file on GitHub
SMI_FILE_URL = "https://raw.githubusercontent.com/silicos-it/PyPharao/master/examples/datasets/compounds_10k.smi"

# Define the local path where the file should be saved
datasets_dir = Path("./datasets")
datasets_dir.mkdir(exist_ok=True) # Create the datasets directory if it doesn't exist
SMI_FILE = datasets_dir / "compounds_10k.smi"

# Download the file
print(f"Downloading {SMI_FILE_URL} to {SMI_FILE}...")
response = requests.get(SMI_FILE_URL)
response.raise_for_status() # Raise an exception for HTTP errors

# Save the file content
with open(SMI_FILE, "wb") as f:
    f.write(response.content)

print(f"Successfully downloaded {SMI_FILE.name} to {SMI_FILE}")

smiles_lines = [
    ln.strip()
    for ln in SMI_FILE.read_text(encoding="utf-8").splitlines()
    if ln.strip()
]
if MAX_COMPOUNDS is not None:
    smiles_lines = smiles_lines[:MAX_COMPOUNDS]

In [ ]:
# Build 3D conformations
NUM_CONFS = 5

print(f"\nBuilding 3D structures for {len(smiles_lines)} SMILES from {SMI_FILE.name}")
prepared: list[tuple[int, str, Chem.Mol]] = []
parse_fail = embed_fail = 0
with tqdm(smiles_lines, desc="3D structures", unit="mol") as pbar:
    for line_idx, smi in enumerate(pbar):
        mol_3d = Chem.MolFromSmiles(smi)
        if mol_3d is None:
            parse_fail += 1
            continue
        mol_3d = Chem.AddHs(mol_3d)
        cids = AllChem.EmbedMultipleConfs(mol_3d, numConfs=NUM_CONFS)
        if len(cids) == 0:
            embed_fail += 1
            continue
        for cid in cids:
            AllChem.MMFFOptimizeMolecule(mol_3d, confId=cid)
        prepared.append((line_idx, smi, mol_3d))
        pbar.set_postfix(ready=len(prepared), parse=parse_fail, embed=embed_fail)

print(
    f"Done: {len(prepared)} ligands ready, "
    f"{parse_fail} invalid SMILES, {embed_fail} embed failures"
)


In [ ]:
# ------------------------------------------------------------
# Run the screen and report the top hits
# ------------------------------------------------------------
#
# `PharmacophoreSearch` enforces excluded volumes on two levels:
#   * `with_exclusion=True`  — soft Pharao volume penalty (subtracted from the
#                              aligned overlap; reflected by `excl_volume`).
#   * `excl_hard_filter=True` (default) — after alignment, every heavy atom is
#                              treated as a sphere of its van der Waals radius;
#                              the hit is rejected if any heavy-atom vdW sphere
#                              swallows a query EXCL marker, i.e. whenever
#                              dist(atom, EXCL) < vdW(atom) + excl_clash_radius.
#                              Pass `excl_hard_filter=False` to recover the pure
#                              soft-penalty behaviour; raise `excl_clash_radius`
#                              to enforce a larger buffer around each EXCL.
#
# The soft penalty is very slow, since all EXCL spheres need to be matched.
# So we will only perform a hard filter here.

hard_searcher = PharmacophoreSearch(query)  # excl_hard_filter=True

print("Hard EXCL atom-clash filter ON (default)")
hard_hits = hard_searcher.screen(prepared, progress=True)
hard_sorted = sort_match_results(hard_hits, sort="descending", key="tanimoto")
print_match_results(hard_sorted, limit=10)

SDF_FILE = Path("./top_hits_phenol_excl.sdf")
write_hits_sdf(hard_sorted[:5], SDF_FILE, pharmacophore=query)
PDB_FILE = Path("./top_hits_phenol_excl.pdb")
write_hits_pdb(hard_sorted[:5], PDB_FILE, pharmacophore=query)


In [ ]:
view = py3Dmol.view()

with open(Path("./paracetamol_with_excl.pdb"), 'r') as f: pdb_data = f.read()
view.addModel(pdb_data, 'pdb')
view.setStyle({'model': 0}, {'line': {}, 'sphere': {'radius': 0.05}})

suppl = Chem.SDMolSupplier(Path("./top_hits_phenol_excl.sdf"))
for mol in suppl:
    if mol is None: continue
    view.addModel(Chem.MolToMolBlock(mol), 'sdf')

view.setStyle({'stick': {'radius': 0.15}, 'sphere': {'radius': 0.2}})
view.show()